# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [1]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 7.8 MB/s eta 0:00:00


## 1) Define KB

In [2]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [6]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = "Looks up relevant information from a custom knowledge base."

    inputs = {
        "query": {"type": "string", "description": "Search query"}
    }

    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "\n".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."

    inputs = {
        "a": {"type": "number", "description": "First number"},
        "b": {"type": "number", "description": "Second number"},
        "op": {"type": "string", "description": "Operation: add or multiply"}
    }

    output_type = "string"

    def forward(self, a: float, b: float, op: str) -> str:
        if op == "multiply":
            return str(a * b)
        elif op == "add":
            return str(a + b)
        else:
            return "Unsupported operation. Use 'add' or 'multiply'."


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

print("Tools ready:", kb_tool.name, math_tool.name)

Tools ready: kb_lookup_tool math_tool


In [4]:
# from smolagents import Tool, TransformersModel, ToolCallingAgent

# class KBLookupTool(Tool):
#     #To-Do: you can customize the name and description of the tool here for example:
#     # name = "kb_lookup_tool"
#     # description = "Looks up relevant information from a custom knowledge base."

#     def __init__(self, kb):
#         super().__init__()
#         self.kb = kb

#     def forward(self, query: str) -> str:
#         q = query.lower()
#         matches = [
#             f"[{item['source']}] {item['text']}"
#             for item in self.kb
#             if any(w in item["text"].lower() for w in q.split())
#         ]
#         return "".join(matches) if matches else "No KB match."


# class MathTool(Tool):
#     name = "math_tool"
#     description = "Add or multiply two numbers."
#     inputs = {
#     "a": {"type": "number", "description": "First number"},
#     "b": {"type": "number", "description": "Second number"},
#     "op": {"type": "string", "description": "Operation: add or multiply"}
# }
#     output_type = "string"

#     def forward(self, a: float, b: float, op: str = "add") -> str:
#         if op == "multiply":
#             return str(a * b)
#         return str(a + b)


# # kb_tool = KBLookupTool(kb_snippets)
# # math_tool = MathTool()

# class KBLookupTool(Tool):
#     name = "kb_lookup_tool"
#     description = "Looks up relevant information from a custom knowledge base."

#     inputs = {
#         "query": {"type": "string", "description": "Search query"}
#     }

#     output_type = "string"

#     def __init__(self, kb):
#         super().__init__()
#         self.kb = kb

#     def forward(self, query: str) -> str:
#         q = query.lower()
#         matches = [
#             f"[{item['source']}] {item['text']}"
#             for item in self.kb
#             if any(w in item["text"].lower() for w in q.split())
#         ]
#         return "\n".join(matches) if matches else "No KB match."

## 3) Model (tiny local)

In [7]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    #To-Do: set up the model parameters as needed
)

print("Model ready:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/smolagents/models.py:933: FutureWarning: The 'model_id' parameter will be required in version 2.0.0. Please update your code to pass this parameter to avoid future errors. For now, it defaults to 'HuggingFaceTB/SmolLM2-1.7B-Instruct'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [8]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are an agentic AI assistant. "
        "Use math_tool for addition or multiplication. "
        "Use kb_lookup_tool for questions about agentic AI, tools, citations, brevity, or missing evidence. "
        "Keep answers short. If you use the knowledge base, include the source tag like [kb:agentic]."
    ),
)

print(agent)

In [ ]:
# agent = ToolCallingAgent(
#     tools=[#To-Do: add your tools here],
#     model=model,
#     max_steps=2,
#     instructions=(
#         #To-Do: you can customize the agent instructions here for example:
#         # "You are an agentic AI that uses tools to answer questions. "
#     ),
# )

# print(agent)

## 5) Test queries

In [10]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q) #To-Do: run the agent on the question q
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - HuggingFaceTB/SmolLM2-1.7B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': 12, 'b': 30, 'op': 'add'}                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: Agentic AI loops plan, choose tools, and reflect before answering. │
│ Useful tools: math, search, and domain-specific lookup. Always cite where evidence came from to stay            │
│ transparent. If evidence is missing, say so and propose a follow-up question.                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Agentic AI loops plan, choose tools, and reflect before answering. Useful tools: math, search, and 
domain-specific lookup. Always cite where evidence came from to stay transparent. If evidence is missing, say so 
and propose a follow-up question.

Final answer: Agentic AI loops plan, choose tools, and reflect before answering. Useful tools: math, search, and 
domain-specific lookup. Always cite where evidence came from to stay transparent. If evidence is missing, say so 
and propose a follow-up question.

[Step 2: Duration 4.42 seconds| Input tokens: 2,482 | Output tokens: 111]

Answer: Agentic AI loops plan, choose tools, and reflect before answering. Useful tools: math, search, and domain-specific lookup. Always cite where evidence came from to stay transparent. If evidence is missing, say so and propose a follow-up question.
